In [1]:
import torch 
import torch.nn.functional as F
torch.set_printoptions(threshold=float('inf'))
torch.set_printoptions(sci_mode=False)
# init memory and buffer
mem_A = torch.zeros(31008, 16)
mem_B = torch.zeros(31008, 16)
mem_w = torch.zeros(585, 16)
reg_b = torch.zeros(5, 16)

In [2]:
model = torch.load('model_Jun21_1216.pt', map_location='cpu')

In [3]:
model['shared_block.conv1.weight']=torch.round(model['shared_block.conv1.weight']*(2**5))
model['shared_block.conv2.weight']=torch.round(model['shared_block.conv2.weight']*(2**6))
model['shared_block.conv3.weight']=torch.round(model['shared_block.conv3.weight']*(2**6))
model['shared_block.conv4.weight']=torch.round(model['shared_block.conv4.weight']*(2**6))
model['shared_block.conv5.weight']=torch.round(model['shared_block.conv5.weight']*(2**6))
model['shared_block.conv1.bias']= torch.round(model['shared_block.conv1.bias']*(2**(4+24)))
model['shared_block.conv2.bias']= torch.round(model['shared_block.conv2.bias']*(2**(4+24)))
model['shared_block.conv3.bias']= torch.round(model['shared_block.conv3.bias']*(2**(4+24)))
model['shared_block.conv4.bias']= torch.round(model['shared_block.conv4.bias']*(2**(4+24)))
model['shared_block.conv5.bias']= torch.round(model['shared_block.conv5.bias']*(2**(4+24)))


In [4]:
model['shared_block.conv1.bias']

tensor([ -6440028.,  37303572., -40756216.,  53731372.,  53949448., -38076176.,
        -62411380., -11106567.,  52830176., -34219148., -59044292.,  53857184.,
        -10175441.,  27525490.,  56935668.,  31598440.])

In [5]:
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8) -> torch.Tensor:
    """
    linear quantization for single fp_tensor
      from
        fp_tensor = (quantized_tensor - zero_point) * scale
      we have,
        quantized_tensor = int(round(fp_tensor / scale)) + zero_point
    :param tensor: [torch.(cuda.)FloatTensor] floating tensor to be quantized
    :param bitwidth: [int] quantization bit width
    :param scale: [torch.(cuda.)FloatTensor] scaling factor
    :param zero_point: [torch.(cuda.)IntTensor] the desired centroid of tensor values
    :return:
        [torch.(cuda.)FloatTensor] quantized tensor whose values are integers
    """
    assert(fp_tensor.dtype == torch.float)
    assert(isinstance(scale, float) or
           (scale.dtype == torch.float and scale.dim() == fp_tensor.dim()))
    assert(isinstance(zero_point, int) or
           (zero_point.dtype == dtype and zero_point.dim() == fp_tensor.dim()))

    # Step 1: scale the fp_tensor
    scaled_tensor = fp_tensor/scale
    #print('scaled tensor: ', scaled_tensor)
    # Step 2: round the floating value to integer value
    rounded_tensor = torch.round(scaled_tensor)

    rounded_tensor = rounded_tensor.to(dtype)
    #print('rounded_Tensor: ', rounded_tensor)
    # Step 3: shift the rounded_tensor to make zero_point 0
    shifted_tensor = rounded_tensor + zero_point
    #print('shifted_tensor: ', shifted_tensor)
    # Step 4: clamp the shifted_tensor to lie in bitwidth-bit range
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)
    #print('quantized_tensor: ', quantized_tensor)
    return quantized_tensor

def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    """
    get quantization scale for single tensor
    :param fp_tensor: [torch.(cuda.)Tensor] floating tensor to be quantized
    :param bitwidth: [int] quantization bit width
    :return:
        [float] scale
        [int] zero_point
    """
    #print(fp_tensor)
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    # hint: one line of code for calculating scale
    scale = (fp_max - fp_min)/(quantized_max - quantized_min)
    if scale == 0:
        scale=1e-9
    
    # hint: one line of code for calculating zero_point
    zero_point = round(quantized_min - fp_min/scale)
    
    # clip the zero_point to fall in [quantized_min, quantized_max]
    if zero_point < quantized_min:
        zero_point = quantized_min
    elif zero_point > quantized_max:
        zero_point = quantized_max
    else: # convert from float to int using round()
        zero_point = round(zero_point)
    return scale, int(zero_point)

def linear_quantize_feature(fp_tensor, bitwidth, dtype=torch.int8): #diff
    """
    linear quantization for feature tensor
    :param fp_tensor: [torch.(cuda.)Tensor] floating feature to be quantized
    :param bitwidth: [int] quantization bit width
    :return:
        [torch.(cuda.)Tensor] quantized tensor
        [float] scale tensor
        [int] zero point
    """
    scale, zero_point = get_quantization_scale_and_zero_point(fp_tensor, bitwidth)
    quantized_tensor = linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype)
    return quantized_tensor, scale, zero_point

def get_quantized_range(bitwidth):
    quantized_max = (1 << (bitwidth - 1)) - 1
    quantized_min = -(1 << (bitwidth - 1))
    return quantized_min, quantized_max

In [6]:
import torch

# ------------------------------------------------------------
# 1)  보조 루틴
# ------------------------------------------------------------
def _quant_range(bitwidth):
    """signed 정수 bit-range (포화 한계값)"""
    q_min = -(1 << (bitwidth - 1))
    q_max =  (1 << (bitwidth - 1)) - 1
    return q_min, q_max


def _check_il(bitwidth, int_len):
    if not (0 <= int_len < bitwidth-1):
        raise ValueError(f"IL={int_len}가 bitwidth={bitwidth}에 부적합")


# ------------------------------------------------------------
# 2)  float → 정수 (Q IL.FL) 양자화
# ------------------------------------------------------------
def quantize_fixed(fp_tensor, bitwidth, int_len, dtype=torch.int32, rounding="nearest"):
    """
    고정소수점 양자화
      Q<IL>.<FL> where FL = bitwidth - IL - 1

    Parameters
    ----------
    fp_tensor : torch.float32/64   – 입력 실수 텐서
    bitwidth  : int                – 총 비트 수 (8,16,32 …)
    int_len   : int                – 정수부 비트 수 (sign 제외)
    dtype     : torch.int{8,16,32} – 반환 정수 dtype
    rounding  : {'nearest','floor'}– 반올림 방식

    Returns
    -------
    q_tensor  : torch.Tensor(dtype)– 양자화된 정수 텐서
    scale     : float              – 실수 ↔ 정수 스케일 (2^FL)
    """

    assert fp_tensor.dtype.is_floating_point, "float 텐서가 필요합니다"
    _check_il(bitwidth, int_len)

    frac_len = bitwidth - int_len - 1   # FL
    scale    = float(1 << frac_len)     # 2^FL

    if rounding == "nearest":
        q = torch.round(fp_tensor * scale)
    elif rounding == "floor":
        q = torch.floor(fp_tensor * scale)
    else:
        raise ValueError("rounding 은 'nearest' 또는 'floor'")

    q_min, q_max = _quant_range(bitwidth)
    q_clamped = torch.clamp(q, q_min, q_max).to(dtype)

    return q_clamped


# ------------------------------------------------------------
# 3)  정수 (Q IL.FL) → float 복원 (디버그용)
# ------------------------------------------------------------
def dequantize_fixed(q_tensor: torch.Tensor,
                     *, scale: float) -> torch.Tensor:
    """int 텐서를 원래 float 근사값으로 복원"""
    return q_tensor.to(torch.float32) / scale


In [7]:
# image & pixel mask
input_img = torch.load('sample_input.pt')[0:3].permute(1, 0, 2, 3) # R, G, B image with size 100 x 100
pixel_mask = torch.load('sample_input.pt')[3:6].permute(1, 0, 2, 3)
#input_img = quantize_fixed(input_img, 8, 2, dtype=torch.int8)
#pixel_mask= quantize_fixed(pixel_mask, 8, 2, dtype=torch.int8)
img_H = 100
img_W = 100
pad_H = 1
pad_W = 1
B, C, H, W = 1, 2, img_H+pad_H*2, img_W+pad_W*2

In [8]:
def padding(memory): # for memory A and B
    address, word = memory.size()
    for addr in range(address):
        # pad vertically
        if (addr%W==0) or (addr%W==W-1): # (addr%102==0) or (addr%102==101)
            memory[addr, :] = torch.zeros(word)
        # pad horizontally
        if (addr >= 0 and addr <= W-1) or ( addr >= (H-1)*W and addr <=H*W) or (addr >= (2*(H-1)*W) and addr <= (2*H*W) ) or ( addr >= 3*(H-1) and addr <= (3*H*W)): # 0~101, 10302~10403, 20604 ~ , 30906~
            memory[addr, :] = torch.zeros(word)
        
            
def padding_after_layer4 (memory):
    address, word = memory.size()
    for addr in range(address):
        # pad vertically
        if (addr%W==0) or ((addr-52)%W==W-1): # (addr%102==0) or (addr-52%102==101)
            memory[addr, :] = torch.zeros(word, dtype=torch.float32)
        # pad horizontally
        if (addr >= 0 and addr <= 52-1) or ( addr >= 2652 and addr <= 2702) or (addr >= 7854 and addr <= 7904) or (addr >= 10506 and addr <= 10556): 
            memory[addr, :] = torch.zeros(word, dtype=torch.float32)

In [9]:
class init_SRAM_write():
    def __call__(self, img, mask):
        # R
        for j in range(100): # H
            for k in range(100): # W
                mem_A[103+j*102+k, 0] = img[0, 0, j:j+1, k:k+1] #channel 1 = img
                mem_A[103+j*102+k, 1] = mask[0, 0, j:j+1, k:k+1] #channel 2 = pixel mask
        # G
        for j in range(100): # H
            for k in range(100): # W
                mem_A[10405+j*102+k, 0] = img[0, 1, j:j+1, k:k+1] #channel 1 = img
                mem_A[10405+j*102+k, 1] = mask[0, 1, j:j+1, k:k+1] #channel 2 = pixel mask
        # B
        for j in range(100): # H
            for k in range(100): # W
                mem_A[20707+j*102+k, 0] = img[0, 2, j:j+1, k:k+1] #channel 1 = img
                mem_A[20707+j*102+k, 1] = mask[0, 2, j:j+1, k:k+1] #channel 2 = pixel mask
        print('memory A write done') 
        
class init_weight_write():
    def __init__(self,):
        self.conv1_weight = model['shared_block.conv1.weight']
        self.conv1_bias = model['shared_block.conv1.bias']
        self.conv2_weight=model['shared_block.conv2.weight']
        self.conv2_bias=model['shared_block.conv2.bias']
        self.conv3_weight=model['shared_block.conv3.weight']
        self.conv3_bias=model['shared_block.conv3.bias']
        self.conv4_weight=model['shared_block.conv4.weight']
        self.conv4_bias=model['shared_block.conv4.bias']
        self.conv5_weight=model['shared_block.conv5.weight']
        self.conv5_bias=model['shared_block.conv5.bias']
    def __call__(self,):
        mem_w[0:144, 0:1]=self.conv1_weight[:, 0, :, :].reshape(144, 1)
        mem_w[0:144, 1:2]=self.conv1_weight[:, 1, :, :].reshape(144, 1)
        mem_w[144:288]=self.conv2_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[288:432]=self.conv3_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[432:576]=self.conv4_weight.permute(0,2,3,1).reshape(144, 16)
        mem_w[576:585]=self.conv5_weight.permute(0,2,3,1).reshape(9, 16)
        reg_b[0, :]=self.conv1_bias
        reg_b[1, :]=self.conv2_bias
        reg_b[2, :]=self.conv3_bias
        reg_b[3, :]=self.conv4_bias
        reg_b[4, 0:1]=self.conv5_bias
        print('weight write done')

widths = {
    1: (9, 2, 1),
    2: (9, 2, 1),
    3: (9, 2, 1),
    4: (9, 2, None),   # max-pool : bias 없음
    5: (9, 2, 1),
    6: (9, 2, 1),      # no-ReLU
}
        
class layer1 ():
    def __call__(self,):
        x_r = mem_A[0:10404, 0:2].reshape(-1, 102, 2).permute(2, 0, 1).unsqueeze(0)
        x_g = mem_A[10302: 20706, 0:2].reshape(-1, 102, 2).permute(2, 0, 1).unsqueeze(0)
        x_b = mem_A[20604:31009, 0:2].reshape(-1, 102, 2).permute(2, 0, 1).unsqueeze(0)
        weight=mem_w[0:144, 0:2].reshape(16, 3, 3, 2).permute(0,3,1,2)
        bias=reg_b[0, :]
                
        y_r = F.relu(F.conv2d(x_r, weight, bias))
        y_g = F.relu(F.conv2d(x_g, weight, bias))
        y_b = F.relu(F.conv2d(x_b, weight, bias))
        y_r=F.pad(y_r, (1,1,1,1))
        y_g=F.pad(y_g, (1,1,1,1))
        y_b=F.pad(y_b, (1,1,1,1)) # 1, 16, 102, 102
        
        mem_B[0:10404, :]=y_r.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[10302:20706, :]=y_g.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[20604:31009, :]=y_b.squeeze().permute(1,2,0).reshape(-1, 16)
        print("layer 1 done")
        
class layer2 ():
    def __call__(self,):
        x_r = mem_B[0:10404, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_g = mem_B[10302: 20706, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_b = mem_B[20604:31009, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        weight=mem_w[144:288].reshape(16, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[1, :]
        
        y_r = F.relu((F.conv2d(x_r, weight, bias)))
        y_g = F.relu((F.conv2d(x_g, weight, bias)))
        y_b = F.relu((F.conv2d(x_b, weight, bias)))
        y_r=F.pad(y_r, (1,1,1,1))
        y_g=F.pad(y_g, (1,1,1,1))
        y_b=F.pad(y_b, (1,1,1,1)) # 1, 16, 102, 102
        mem_A[0:10404, :]=y_r.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_A[10302:20706, :]=y_g.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_A[20604:31009, :]=y_b.squeeze().permute(1,2,0).reshape(-1, 16)
        print("layer 2 done")
        
class layer3 ():
    def __call__(self,):
        x_r = mem_A[0:10404, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_g = mem_A[10302: 20706, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        x_b = mem_A[20604:31009, :].reshape(-1, 102, 16).permute(2,0,1).unsqueeze(0)
        weight=mem_w[288:432].reshape(16, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[2, :]
        
        y_r = F.relu((F.conv2d(x_r, weight, bias)))
        y_g = F.relu((F.conv2d(x_g, weight, bias)))
        y_b = F.relu((F.conv2d(x_b, weight, bias)))
        y_r=F.pad(y_r, (1,1,1,1))
        y_g=F.pad(y_g, (1,1,1,1))
        y_b=F.pad(y_b, (1,1,1,1))
        mem_B[0:10404, :]=y_r.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[10302:20706, :]=y_g.squeeze().permute(1,2,0).reshape(-1, 16)
        mem_B[20604:31009, :]=y_b.squeeze().permute(1,2,0).reshape(-1, 16)
        print("layer 3 done")
        
class layer4 (): # maxpool
    def __call__(self,):
        x_r = mem_B[0:10404, :].reshape(-1, 102, 16)[1:101, 1:101, :].permute(2,0,1).unsqueeze(0)
        x_g = mem_B[10302: 20706, :].reshape(-1, 102, 16)[1:101, 1:101, :].permute(2,0,1).unsqueeze(0)
        x_b = mem_B[20604:31009, :].reshape(-1, 102, 16)[1:101, 1:101, :].permute(2,0,1).unsqueeze(0)
        print(x_r.shape, x_g.shape, x_b.shape)
        
        y_r=F.max_pool2d(x_r, kernel_size=(4,2), stride=(4,2)).to(torch.float32) #1, 16, 50, 25
        y_g=F.max_pool2d(x_g, kernel_size=2, stride=2).to(torch.float32)         #1, 16, 50, 50
        y_b=F.max_pool2d(x_b, kernel_size=(4,2), stride=(4,2)).to(torch.float32) #1, 16, 50, 25
        y_r = F.relu(y_r)
        y_g = F.relu(y_g)
        y_b = F.relu(y_b)

        for i in range(25):
            mem_A[103+i*102:153+i*102, :]=y_r.squeeze().permute(1,2,0)[i, :, :]
        
        for i in range(50):
            mem_A[2755+i*102:2805+i*102, :]=y_g.squeeze().permute(1,2,0)[i, :, :]
            
        for i in range(25):
            mem_A[7957+i*102:8007+i*102, :]=y_b.squeeze().permute(1,2,0)[i, :, :]
        print("layer 4 done")
        
        
class layer5 ():
    def __call__(self,):
        padding_after_layer4(mem_A)
        x_r = torch.zeros(1, 16, 52, 27, dtype=torch.float32)
        for i in range(27):
            x_r[0, :, :, i] = mem_A[0+102*i:52+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_g = torch.zeros(1, 16, 52, 52, dtype=torch.float32)
        for i in range(52):
            x_g[0, :, :, i] = mem_A[2652+102*i:2704+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_b = torch.zeros(1, 16, 52, 27, dtype=torch.float32)
        for i in range(27):
            x_b[0, :, :, i] = mem_A[7854+102*i:7906+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        weight=mem_w[432:576].reshape(16, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[3, :]
        
        y_r=F.relu((F.conv2d(x_r, weight, bias)))
        y_g=F.relu((F.conv2d(x_g, weight, bias)))
        y_b=F.relu((F.conv2d(x_b, weight, bias)))
        print(y_r.shape, y_g.shape, y_b.shape)
        for i in range(25):
            mem_B[103+i*102:153+i*102, :]=y_r.squeeze().permute(1,2,0).transpose(0,1)[i, :, :]
        
        for i in range(50):
            mem_B[2754+i*102:2804+i*102, :]=y_g.squeeze().permute(1,2,0).transpose(0,1)[i, :, :]
            
        for i in range(25):
            mem_B[7956+i*102:8006+i*102, :]=y_b.squeeze().permute(1,2,0).transpose(0,1)[i, :, :]
        print("layer 5 done")
        
class layer6 ():
    def __call__(self,):
        
        padding_after_layer4(mem_B)
        x_r = torch.zeros(1, 16, 52, 27, dtype=torch.float32)
        for i in range(27):
            x_r[0, :, :, i] = mem_B[0+102*i:52+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_g = torch.zeros(1, 16, 52, 52, dtype=torch.float32)
        for i in range(52):
            x_g[0, :, :, i] = mem_B[2652+102*i:2704+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        x_b = torch.zeros(1, 16, 52, 27, dtype=torch.float32)
        for i in range(27):
            x_b[0, :, :, i] = mem_B[7854+102*i:7906+102*i, :].permute(1, 0).reshape(16, 52).unsqueeze(0)
        weight=mem_w[576:585].reshape(1, 3, 3, 16).permute(0, 3, 1, 2)
        bias=reg_b[4, 0:1]
        
        y_r=F.relu(F.conv2d(x_r, weight, bias))
        y_g=F.relu(F.conv2d(x_g, weight, bias))
        y_b=F.relu(F.conv2d(x_b, weight, bias))
        print(y_r.shape, y_g.shape, y_b.shape)
        for i in range(25):
            mem_A[103+i*102:153+i*102, 0:1]=y_r.squeeze(0).permute(1,2,0).transpose(0,1)[i, :, :]
        
        for i in range(50):
            mem_A[2754+i*102:2804+i*102, 0:1]=y_g.squeeze(0).permute(1,2,0).transpose(0,1)[i, :, :]
            
        for i in range(25):
            mem_A[7956+i*102:8006+i*102, 0:1]=y_b.squeeze(0).permute(1,2,0).transpose(0,1)[i, :, :]
        print("layer 6 done")


In [10]:
def check(memory):
    x_r = memory[0:10404, :].reshape(-1, 102, 16).permute(2, 0, 1)
    x_g = memory[10302: 20706, :].reshape(-1, 102, 16).permute(2, 0, 1)
    x_b = memory[20604:31009, :].reshape(-1, 102, 16).permute(2, 0, 1)
    return torch.cat([x_r, x_g, x_b])[:, 1:101, 1:101].unsqueeze(0).reshape(3, 16, 100, 100).to('cuda')

In [11]:
def check_after_layer4(memory):
    x_r = torch.zeros(1, 1, 52, 27, dtype=torch.float32)
    for i in range(27):
        x_r[0, :, :, i] = memory[0+102*i:52+102*i, 0:1].permute(1, 0).reshape(1, 52).unsqueeze(0)
    x_g = torch.zeros(1, 1, 52, 52, dtype=torch.float32)
    for i in range(52):
        x_g[0, :, :, i] = memory[2652+102*i:2704+102*i, 0:1].permute(1, 0).reshape(1, 52).unsqueeze(0)
    x_b = torch.zeros(1, 1, 52, 27, dtype=torch.float32)
    for i in range(27):
        x_b[0, :, :, i] = memory[7854+102*i:7906+102*i, 0:1].permute(1, 0).reshape(1, 52).unsqueeze(0)
    result_r = x_r[:, :, 1:51, 1:26]
    result_g = x_g[:, :, 1:51, 1:51]
    result_b = x_b[:, :, 1:51, 1:26]
    print(result_r, result_g, result_b)

In [12]:
def save_txt(tensor, file_name):
    import numpy as np
    assert tensor.dim()==2
    tensor=np.array(tensor).astype(int).astype(str).tolist()
    with open(file_name, "w") as file:
        for row in tensor:
            file.write(" ".join(row) + "\n")

In [13]:
# init layer classes
layer1_en = layer1()
layer2_en = layer2()
layer3_en = layer3()
layer4_en = layer4()
layer5_en = layer5()
layer6_en = layer6()
SRAM_write = init_SRAM_write()
weight_write = init_weight_write()

# Main FSM 
#IDLE
print("Process Started")

#S_SRAM_W
SRAM_write(input_img, pixel_mask)
weight_write()
#reg_b = quantize_fixed(reg_b, 32, 1, dtype=torch.int32)
#mem_w = quantize_fixed(mem_w, 8, 1, dtype=torch.float32)

# #Layer 1~6
# layer1_en()
# mem_B = torch.clamp(torch.floor(mem_B*(2**4)), -127, 128)
# check(mem_B)
# save_txt(torch.clamp(torch.floor(mem_B*(2**4)), -127, 128), 'after_layer1_memB.txt')
# layer2_en()
# save_txt(torch.clamp(torch.floor(mem_A*(2**4)), -127, 128), 'after_layer2_memA.txt')
# layer3_en()
# save_txt(torch.clamp(torch.floor(mem_B*(2**4)), -127, 128), 'after_layer3_memB.txt')
# layer4_en()
# save_txt(torch.clamp(torch.floor(mem_A*(2**4)), -127, 128), 'after_layer4_memA.txt')
# layer5_en()
# save_txt(torch.clamp(torch.floor(mem_B*(2**4)), -127, 128), 'after_layer5_memB.txt')
# layer6_en()
# save_txt(torch.clamp(torch.floor(mem_A*(2**4)), -127, 128), 'after_layer6_memA.txt')

# #IDLE
# print("Process Finished")

Process Started
memory A write done
weight write done


In [14]:
import torch

def compare_integer_tensors_allow_1_diff(t1: torch.Tensor, t2: torch.Tensor):
    """
    Compare two integer tensors and return mismatched indices and values,
    allowing a difference of up to ±1.
    """
    if t1.shape != t2.shape:
        raise ValueError(f"Shape mismatch: {t1.shape} vs {t2.shape}")

    # Compute absolute difference
    diff = (t1 - t2).abs()
    
    # Boolean mask where difference is greater than 1
    threshold=1e-2
    diff_mask = diff > threshold
    # Get indices where difference exceeds allowed margin
    indices = diff_mask.nonzero(as_tuple=False)
    
    # Collect mismatches
    mismatches = []
    for idx in indices:
        idx_tuple = tuple(idx.tolist())
        val1 = t1[idx_tuple].item()
        val2 = t2[idx_tuple].item()
        mismatches.append((idx_tuple, val1, val2))
    print(f"🔍 Total mismatches (|diff| > {threshold}): {len(mismatches)}")
    return mismatches




In [15]:
import torch

def txt_to_tensor(path, pad_value=0, dtype=torch.long):
    """
    Parameters
    ----------
    path : str
        읽어들일 텍스트 파일 경로
    pad_value : int, optional
        행마다 원소 개수가 다를 때 채워 넣을 값(기본 0)
    dtype : torch.dtype, optional
        생성할 텐서 자료형(기본 torch.long)

    Returns
    -------
    torch.Tensor
        2-차원 Tensor (행 = 파일의 유효 라인 수, 열 = 최장 라인의 원소 개수)
    """
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            # 행 번호와 값을 나누기 위해 ':' 기준 split
            if ":" not in line:
                continue  # 빈 줄 등은 건너뜀

            _, values_str = line.split(":", 1)        # ':' 앞쪽(행 번호)은 버림
            values = values_str.strip().split()       # 공백으로 숫자 분리

            if not values:                            # 값이 없는 라인은 skip
                continue

            rows.append([int(v) for v in values])

    if not rows:
        raise ValueError("파일에서 숫자를 찾지 못했습니다.")

    # 가장 긴 행 길이에 맞춰 패딩(pad_value) 삽입
    max_len = max(len(r) for r in rows)
    padded = [r + [pad_value] * (max_len - len(r)) for r in rows]

    return torch.tensor(padded, dtype=dtype)


# 사용 예시
tensor2d = txt_to_tensor("text/feature_data_out_1.txt")   # 경로를 여러분의 txt 파일로 바꿔주세요
print(tensor2d.shape)                    # ex) torch.Size([35, 15])
print(tensor2d)


torch.Size([31008, 16])
tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  

In [25]:
mem_A = torch.round(mem_A*(2**4)).to(torch.int8)

In [17]:
compare_integer_tensors_allow_1_diff(tensor2d, mem_B)

🔍 Total mismatches (|diff| > 0.01): 324272


[((103, 1), 4, 0.0),
 ((103, 2), 11, 0.0),
 ((103, 4), 1, 0.0),
 ((103, 5), 14, 0.0),
 ((103, 6), 2, 0.0),
 ((103, 8), 9, 0.0),
 ((103, 9), 9, 0.0),
 ((103, 11), 2, 0.0),
 ((103, 13), 13, 0.0),
 ((103, 14), 1, 0.0),
 ((103, 15), 9, 0.0),
 ((104, 1), 2, 0.0),
 ((104, 2), 6, 0.0),
 ((104, 4), 4, 0.0),
 ((104, 8), 11, 0.0),
 ((104, 10), 6, 0.0),
 ((104, 11), 6, 0.0),
 ((104, 12), 7, 0.0),
 ((104, 13), 14, 0.0),
 ((104, 15), 8, 0.0),
 ((105, 0), 8, 0.0),
 ((105, 2), 7, 0.0),
 ((105, 5), 8, 0.0),
 ((105, 6), 6, 0.0),
 ((105, 7), 4, 0.0),
 ((105, 8), 11, 0.0),
 ((105, 9), 4, 0.0),
 ((105, 11), 7, 0.0),
 ((105, 13), 13, 0.0),
 ((105, 15), 7, 0.0),
 ((106, 0), 1, 0.0),
 ((106, 2), 8, 0.0),
 ((106, 4), 1, 0.0),
 ((106, 5), 7, 0.0),
 ((106, 8), 5, 0.0),
 ((106, 9), 6, 0.0),
 ((106, 12), 2, 0.0),
 ((106, 13), 5, 0.0),
 ((106, 15), 3, 0.0),
 ((107, 1), 4, 0.0),
 ((107, 2), 12, 0.0),
 ((107, 5), 16, 0.0),
 ((107, 8), 8, 0.0),
 ((107, 9), 9, 0.0),
 ((107, 11), 3, 0.0),
 ((107, 13), 13, 0.0),
 ((107,

In [18]:
import torch

def fp32_to_int32_hex8(int_tensor: torch.Tensor) -> str:
    """
    fp_tensor : torch.float32 텐서
    scale     : 양자화 스케일 (예: (2**31 - 1) / fp_tensor.abs().max())
    리턴값    : 8-자리 헥사 문자열을 16개씩 묶어 줄바꿈한 덤프
    """
    # # 1) FP32 → INT32 양자화
    # int_tensor = torch.round(fp_tensor * scale)            \
    #                    .clamp(-2**31, 2**31 - 1)           \
    #                    .to(torch.int32)

    # 2) INT32 → 8-자리 헥사(32 bit, two's complement)
    hex_words = [format(x.item() & 0xFFFFFFFF, '08X') 
                 for x in int_tensor.view(-1)]

    # 3) 보기 좋게 16 word(= 64 byte)마다 줄바꿈
    WORDS_PER_LINE = 16
    lines = [
        "".join(hex_words[i*WORDS_PER_LINE : (i+1)*WORDS_PER_LINE])
        for i in range((len(hex_words)+WORDS_PER_LINE-1)//WORDS_PER_LINE)
    ]
    return "\n".join(lines)


In [19]:
def int_to_hex(int_tensor): #print the data in hexa format
    int_tensor=int_tensor.int()
    # Step 5: Convert to binary two's complement representation (optional for display)
    binary_tensor = [format(x & 0xFF, '08b') for x in int_tensor.view(-1).tolist()]
    hex_tensor=[ hex(int(binary_str[:4], 2))[2:].upper()+hex(int(binary_str[4:], 2))[2:].upper() for binary_str in binary_tensor]
    output = "\n".join("".join(hex_tensor[i * 16:(i + 1) * 16]) for i in range(5)) #16 num, 210 lines -> 32 x 210

    return output

print(int_to_hex(reg_b))

A414082C08F08CF9E0743CA02F72F468
99E628676C5C7F39C20CDEA4F22D3DA4
19F989FCF803F3887D365A9D89B085F1
9865ACED3B8D4B9B133034CE150115CF
C4000000000000000000000000000000


In [20]:
def int32_to_hex(int_tensor, words_per_line=4):
    """
    Flatten `int_tensor`, treat each entry as a signed 32-bit int,
    and print as 8-digit uppercase hex words, grouping `words_per_line` per output line.
    """
    # ensure integer type
    int_tensor = int_tensor.int()
    
    # flatten to Python list
    vals = int_tensor.view(-1).tolist()
    
    # format each value as zero-padded 8-digit hex (two’s-complement)
    hex_vals = [format(x & 0xFFFFFFFF, '08X') for x in vals]
    
    # group into lines
    lines = []
    for i in range(0, len(hex_vals), words_per_line):
        chunk = hex_vals[i : i + words_per_line]
        lines.append(''.join(chunk))
    
    return "\n".join(lines)
print(int32_to_hex(reg_b, words_per_line=16)) # 16 words per line

FF9DBBA402393514FD921C080333E02C03373408FDBB00F0FC47AD8CFF5686F903261FE0FDF5DB74FC7B0E3C0335CBA0FF64BC2F01A401720364C4F401E22768
FF02EC99002AA2E60145DF28007C7667010A6F6C006E3C5CFF01167FFF451A39FF197BC2FEFF9E0C005EAFDE010C9FA40134A2F200A8232DFFA2CA3D00BB63A4
FFBD8B19FFE982F9FFC21A89FFF07CFC01243DF8FF3D7903FFC303F3FF0F0188FF62937D01343E3600D5ED5A00F3459DFFF5A689FED28CB000D35C85FF005DF1
FF6E3A98FF4FE865014A41ACFF4927ED009A5F3B00B4698D001BC84BFF47EF9BFF011F13004C153000976E340124BFCEFF4F9215FF500C01003B1115FF6D9FCF
0345B2C4000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


In [39]:
#!/usr/bin/env python3
"""
generate_bram_init.py

– data: initialize할 8bit 값들의 파이썬 시퀀스 또는 torch.Tensor (dtype=torch.int8)
– radix: COE 파일에 쓸 진법 (2, 10, 16 중 선택)
– per_line: 한 줄에 묶을 요소 개수 (여기선 32)
"""

import torch
from typing import Union, Sequence, Any

def flatten(seq):
    out: list[int] = []
    for x in seq:
        if isinstance(x, (list, tuple)):
            out.extend(flatten(x))
        else:
            out.append(int(x))
    return out

def extract_raw(data):
    if isinstance(data, torch.Tensor):
        assert data.dtype == torch.int8, "Tensor dtype은 torch.int8이어야 합니다."
        raw = data.flatten().tolist()
    else:
        raw = flatten(data)
    return raw

def fmt_u8(val: int, radix: int) -> str:
    u8 = val & 0xFF
    if radix == 16:
        return format(u8, '02X')
    elif radix == 2:
        return format(u8, '08b')
    else:
        return str(u8)

def write_coe_file(
    data,
    filename: str = "init.coe",
    radix: int = 16,
    per_line: int = 16,
):
    assert radix in (2, 10, 16), "radix는 2, 10, 16 중 하나여야 합니다."
    raw_vals = extract_raw(data)
    fmt_vals = [fmt_u8(v, radix) for v in raw_vals]
    n = len(fmt_vals)

    with open(filename, 'w') as f:
        f.write(f"memory_initialization_radix = {radix};\n")
        f.write("memory_initialization_vector =\n")
        for i in range(0, n, per_line):
            chunk = fmt_vals[i : i + per_line]
            line_str = "".join(chunk)  # 값들 사이에 구분자 없이 바로 붙이기
            # 마지막 줄인지 여부 판단
            if i + per_line < n:
                f.write(line_str + ",\n")  # 줄 끝에만 쉼표
            else:
                f.write(line_str + ";\n")  # 마지막 줄엔 세미콜론

    print(f"Generated {filename}")



In [37]:
mem_A

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  

In [45]:
reg_b = reg_b.to(torch.int8)  # mem_B를 int8로 변환
write_coe_file(reg_b, filename="reg_b.coe", radix=16)

Generated reg_b.coe


In [42]:
mem_w

tensor([[  1,   2,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  1,  -5,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  4,   7,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [ -6,   8,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  5,   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  1,  -1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [ -6,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [ -5,  -5,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [ -3,  -2,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [ -1,   7,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [ 